# HearAssist — Benchmark sonore YAMNet et EfficientAT

Exécuter les cellules dans l'ordre dans un **nouveau notebook Kaggle**, avec **Internet activé**.
Le CPU suffit. Le réglage par défaut utilise le CPU pour les deux modèles, avec deux threads.

Ce notebook télécharge ESC-50 et les poids préentraînés, compare huit sons utiles,
affiche les scores et les erreurs avec lecteurs audio, puis exporte une archive.
Il teste la classification de fichiers et propose une simulation par fenêtres en fin de notebook.
La latence du microphone et les fausses alertes par heure devront être mesurées dans l'application.

Choix du protocole

- 8 catégories cibles × 40 fichiers = 320 fichiers.
- 4 catégories de fond × 40 fichiers = 160 fichiers supplémentaires pour tester les fausses détections.
- Folds 1–2 pour régler les seuils, folds 3–5 pour le test. Aucun entraînement.
- Les scores natifs sont regroupés selon une correspondance explicite, identique pour les deux modèles.
- Les poids choisis sont préentraînés sur AudioSet. L'absence de recoupement entre tous les corpus historiques n'est pas établie ici.

ESC-50 est distribué sous licence CC BY-NC 3.0, avec des conditions distinctes pour ESC-10.
Consulter les licences avant une réutilisation commerciale des données.


## 1. Installer les dépendances

Cette cellule conserve PyTorch, torchvision et torchaudio fournis par Kaggle.
YAMNet utilise sa version officielle TFLite avec LiteRT sur CPU.
Après un redémarrage de session, reprendre à la cellule 2 si l'installation a réussi.


In [ ]:
import subprocess
import sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ai-edge-litert==2.3.0', 'soundfile>=0.12', 'scipy>=1.11',
    'pandas>=2.0', 'scikit-learn>=1.3', 'matplotlib>=3.7', 'tqdm>=4.65'
], check=True)


## 2. Configuration


In [ ]:
import os
import sys
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
from pathlib import Path
import json
import platform
import importlib.metadata

ROOT = Path('/kaggle/working/hearassist_sound')
ROOT.mkdir(parents=True, exist_ok=True)
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
EA_DEVICE = 'cpu'  # 'cuda' possible si un GPU est activé. YAMNet reste sur CPU.
THREADS = 2
EA_CHECKPOINT = 'mn10_as'
EA_REVISION = 'a425fdce92572e602a1d5634799bd9f1f2efa806'
CALIBRATION_FOLDS = [1, 2]
TEST_FOLDS = [3, 4, 5]
MAX_PER_CLASS = None  # 5 pour un essai rapide, None pour les 40 fichiers par catégorie.
N_AUDIO_DISPLAY = 10

print('Python', platform.python_version())
print('Sorties', RESULTS)


## 3. Télécharger le code et le dataset

Aucun fichier à ajouter via Add Input. Le téléchargement d'ESC-50 est d'environ 600 Mo.
Si ESC-50 est déjà ajouté dans /kaggle/input, le notebook réutilise cette copie.
Les téléchargements sont réutilisés à l'exécution suivante dans la même session.


In [ ]:
import urllib.request
import zipfile
import shutil
import time

SOURCES = {
    'efficientat': f'https://codeload.github.com/fschmid56/EfficientAT/zip/{EA_REVISION}',
    'esc50': 'https://codeload.github.com/karolpiczak/ESC-50/zip/33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6',
    'yamnet': 'https://storage.googleapis.com/mediapipe-models/audio_classifier/yamnet/float32/1/yamnet.tflite',
}

def download(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    temporary = destination.with_suffix(destination.suffix + '.part')
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={'User-Agent': 'HearAssist-benchmark'})
            with urllib.request.urlopen(request, timeout=120) as source, temporary.open('wb') as target:
                shutil.copyfileobj(source, target)
            temporary.replace(destination)
            return destination
        except Exception:
            temporary.unlink(missing_ok=True)
            if attempt == 2:
                raise
            time.sleep(2)

def extract_archive(archive, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            resolved = (destination / member.filename).resolve()
            if not resolved.is_relative_to(destination.resolve()):
                raise ValueError('Chemin non autorisé dans une archive.')
        bundle.extractall(destination)

EA_PARENT = ROOT / 'efficientat_source'
if not list(EA_PARENT.glob('*/models/mn/model.py')):
    extract_archive(download(SOURCES['efficientat'], ROOT / 'efficientat.zip'), EA_PARENT)
EA_ROOT = next(EA_PARENT.glob('*/models/mn/model.py')).parents[2]

candidates = list(Path('/kaggle/input').rglob('esc50.csv')) if Path('/kaggle/input').exists() else []
candidates += list((ROOT / 'esc50_source').glob('*/meta/esc50.csv'))
metadata_path = next((p for p in candidates if (p.parent.parent / 'audio').is_dir()), None)
if metadata_path is None:
    print('Téléchargement ESC-50…', flush=True)
    extract_archive(download(SOURCES['esc50'], ROOT / 'esc50.zip'), ROOT / 'esc50_source')
    metadata_path = next((ROOT / 'esc50_source').glob('*/meta/esc50.csv'))
ESC_ROOT = metadata_path.parent.parent
YAMNET_PATH = download(SOURCES['yamnet'], ROOT / 'yamnet.tflite')
print('Dataset', ESC_ROOT)
print('Code EfficientAT', EA_ROOT)


## 4. Sélection et correspondance des catégories

Les groupes ci-dessous sont une convention de benchmark à examiner avant toute publication.
Par exemple « Shatter » représente ici le bris de verre et « Knock » les coups à la porte.
Pour un groupe réunissant plusieurs sorties natives, on retient leur score maximum.
Le nom anglais natif le mieux classé sera également conservé dans les résultats.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

TARGETS = {
    'dog': {'fr': 'Aboiement', 'labels': ['Dog', 'Bark']},
    'crying_baby': {'fr': 'Pleurs de bébé', 'labels': ['Baby cry, infant cry']},
    'door_wood_knock': {'fr': 'Coups à la porte', 'labels': ['Knock']},
    'siren': {'fr': 'Sirène', 'labels': ['Siren', 'Police car (siren)', 'Ambulance (siren)', 'Fire engine, fire truck (siren)', 'Civil defense siren']},
    'car_horn': {'fr': 'Klaxon', 'labels': ['Vehicle horn, car horn, honking']},
    'clock_alarm': {'fr': 'Réveil', 'labels': ['Alarm clock']},
    'glass_breaking': {'fr': 'Bris de verre', 'labels': ['Shatter']},
    'engine': {'fr': 'Moteur', 'labels': ['Engine', 'Light engine (high frequency)', 'Medium engine (mid frequency)', 'Heavy engine (low frequency)', 'Engine starting', 'Engine knocking']},
}
BACKGROUND = ['rain', 'clapping', 'footsteps', 'keyboard_typing']
CATEGORIES = list(TARGETS)
metadata = pd.read_csv(metadata_path)
selected = metadata[metadata.category.isin(CATEGORIES + BACKGROUND)].copy()
selected = selected.sort_values(['category', 'fold', 'filename'])
if MAX_PER_CLASS is not None:
    # Répartir l'essai rapide sur les deux partitions.
    parts = []
    for category, group in selected.groupby('category'):
        calibration = group[group.fold.isin(CALIBRATION_FOLDS)].head(max(1, MAX_PER_CLASS // 2))
        test = group[group.fold.isin(TEST_FOLDS)].head(max(1, MAX_PER_CLASS - len(calibration)))
        parts.extend([calibration, test])
    selected = pd.concat(parts).sort_values(['category', 'fold', 'filename'])
selected['partition'] = np.where(selected.fold.isin(CALIBRATION_FOLDS), 'calibration', 'test')
selected['is_target'] = selected.category.isin(CATEGORIES)
selected = selected.reset_index(drop=True)
assert set(selected.category) == set(CATEGORIES + BACKGROUND)
assert all((ESC_ROOT / 'audio' / name).is_file() for name in selected.filename)
selected.to_csv(RESULTS / 'manifest.csv', index=False)
display(pd.crosstab(selected.category, selected.partition))
display(pd.DataFrame([{'categorie': key, 'affichage': value['fr'], 'sorties_regroupees': ', '.join(value['labels'])} for key, value in TARGETS.items()]))
print('Nombre total de fichiers', len(selected))


## 5. Charger les modèles

Deux moteurs différents sont utilisés. Le temps mesuré décrit cette configuration complète.
Avec EA_DEVICE='cpu', les deux modèles utilisent le CPU en float32, avec deux threads.
La comparaison des temps sur CPU et GPU devra être identifiée comme telle si tu modifies EA_DEVICE.


In [ ]:
import torch
import torchvision
import torchaudio
from ai_edge_litert.interpreter import Interpreter

if EA_DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError("GPU absent. Active un GPU Kaggle ou remets EA_DEVICE = 'cpu'.")
torch.set_num_threads(THREADS)
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__, '| torchaudio', torchaudio.__version__)
print('EfficientAT', EA_DEVICE, '| YAMNet CPU')

# Les étiquettes intégrées au fichier officiel suivent directement ses sorties.
with zipfile.ZipFile(YAMNET_PATH) as bundle:
    yamnet_labels = bundle.read('yamnet_label_list.txt').decode('utf-8').splitlines()
yamnet = Interpreter(model_path=str(YAMNET_PATH), num_threads=THREADS)
yamnet.allocate_tensors()
y_input = yamnet.get_input_details()[0]
y_output = yamnet.get_output_details()[0]
YAMNET_SAMPLES = int(np.prod(y_input['shape']))
assert y_output['shape'][-1] == len(yamnet_labels) == 521

# Le dépôt EfficientAT utilise des chemins de métadonnées relatifs.
os.chdir(EA_ROOT)
sys.path.insert(0, str(EA_ROOT))
from models.mn.model import get_model
from models.preprocess import AugmentMelSTFT
from helpers.utils import NAME_TO_WIDTH, labels as efficientat_labels

efficientat = get_model(pretrained_name=EA_CHECKPOINT, width_mult=NAME_TO_WIDTH(EA_CHECKPOINT))
efficientat = efficientat.to(EA_DEVICE).eval()
mel = AugmentMelSTFT(n_mels=128, sr=32000, win_length=800, hopsize=320).to(EA_DEVICE).eval()
assert len(efficientat_labels) == 527

MODEL_LABELS = {'YAMNet': yamnet_labels, 'EfficientAT': efficientat_labels}
GROUP_INDICES = {}
for model_name, labels in MODEL_LABELS.items():
    GROUP_INDICES[model_name] = []
    for category in CATEGORIES:
        missing = set(TARGETS[category]['labels']) - set(labels)
        if missing:
            raise ValueError(f'{model_name} / {category} : étiquettes absentes {missing}')
        GROUP_INDICES[model_name].append([labels.index(label) for label in TARGETS[category]['labels']])
print('Deux modèles chargés. Fenêtre YAMNet', YAMNET_SAMPLES / 16000, 's')


## 6. Prétraitement et fonctions de prédiction

Le même fichier est rééchantillonné à 16 kHz pour YAMNet et à 32 kHz pour EfficientAT.
Aucune normalisation du volume par fichier. YAMNet est appliqué par fenêtres avec chevauchement,
puis ses scores sont moyennés. EfficientAT reçoit le fichier complet.
Ces différences font partie du protocole et sont conservées dans les exports.


In [ ]:
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd
from time import perf_counter

def load_audio(path, sample_rate):
    samples, original_rate = sf.read(path, dtype='float32', always_2d=True)
    samples = samples.mean(axis=1)
    if original_rate != sample_rate:
        factor = gcd(int(original_rate), sample_rate)
        samples = resample_poly(samples, sample_rate // factor, int(original_rate) // factor)
    return np.ascontiguousarray(samples, dtype=np.float32)

def predict_yamnet(samples):
    hop = 7680  # 0,48 seconde à 16 kHz.
    starts = list(range(0, max(len(samples) - YAMNET_SAMPLES, 0) + 1, hop))
    last = max(0, len(samples) - YAMNET_SAMPLES)
    if starts[-1] != last:
        starts.append(last)
    scores = []
    for start in starts:
        window = samples[start:start + YAMNET_SAMPLES]
        window = np.pad(window, (0, YAMNET_SAMPLES - len(window))).astype(np.float32)
        yamnet.set_tensor(y_input['index'], window.reshape(y_input['shape']))
        yamnet.invoke()
        scores.append(yamnet.get_tensor(y_output['index']).reshape(-1).copy())
    return np.mean(scores, axis=0)

def predict_efficientat(samples):
    waveform = torch.from_numpy(samples[None, :]).to(EA_DEVICE)
    with torch.inference_mode():
        spectrogram = mel(waveform)
        logits, _ = efficientat(spectrogram.unsqueeze(0))
        return torch.sigmoid(logits.float()).squeeze(0).cpu().numpy()

PREDICTORS = {'YAMNet': predict_yamnet, 'EfficientAT': predict_efficientat}
SAMPLE_RATES = {'YAMNet': 16000, 'EfficientAT': 32000}

def grouped_scores(model_name, native_scores):
    return np.array([native_scores[indexes].max() for indexes in GROUP_INDICES[model_name]])

# Chauffe des moteurs pour écarter le premier appel du chronométrage principal.
for model_name in PREDICTORS:
    wave = load_audio(ESC_ROOT / 'audio' / selected.iloc[0].filename, SAMPLE_RATES[model_name])
    result = PREDICTORS[model_name](wave)
    assert result.shape == (len(MODEL_LABELS[model_name]),)
    assert np.isfinite(result).all()
    print(model_name, 'OK', MODEL_LABELS[model_name][int(result.argmax())])


## 7. Exécuter le benchmark

Les deux modèles sont évalués successivement. Le chronométrage comprend le traitement de l'audio
rééchantillonné et l'inférence, mais exclut la lecture du fichier, le rééchantillonnage,
le téléchargement, le chargement et la chauffe. Sur GPU, les calculs sont synchronisés.


In [ ]:
from tqdm.auto import tqdm
import hashlib

records = []
for model_name, predictor in PREDICTORS.items():
    model_records = []
    rate = SAMPLE_RATES[model_name]
    for example in tqdm(selected.to_dict('records'), desc=model_name):
        samples = load_audio(ESC_ROOT / 'audio' / example['filename'], rate)
        if model_name == 'EfficientAT' and EA_DEVICE == 'cuda':
            torch.cuda.synchronize()
        start = perf_counter()
        native = predictor(samples)
        if model_name == 'EfficientAT' and EA_DEVICE == 'cuda':
            torch.cuda.synchronize()
        elapsed = perf_counter() - start
        scores = grouped_scores(model_name, native)
        top = np.argsort(native)[::-1][:3]
        row = {
            'model': model_name, 'filename': example['filename'],
            'reference': example['category'], 'fold': example['fold'],
            'partition': example['partition'], 'is_target': example['is_target'],
            'prediction_8_classes': CATEGORIES[int(scores.argmax())],
            'native_top3': ' | '.join(f'{MODEL_LABELS[model_name][i]} ({native[i]:.3f})' for i in top),
            'seconds': elapsed, 'audio_seconds': len(samples) / rate,
            'device': EA_DEVICE if model_name == 'EfficientAT' else 'cpu',
        }
        row.update({f'score_{category}': float(scores[i]) for i, category in enumerate(CATEGORIES)})
        model_records.append(row)
    pd.DataFrame(model_records).to_csv(RESULTS / f'{model_name.lower()}_predictions.csv', index=False)
    records.extend(model_records)

predictions = pd.DataFrame(records)
versions = {}
for package in ['numpy', 'torch', 'torchvision', 'torchaudio', 'ai-edge-litert', 'scipy', 'pandas', 'scikit-learn']:
    versions[package] = importlib.metadata.version(package)
config = {
    'targets': TARGETS, 'background': BACKGROUND,
    'calibration_folds': CALIBRATION_FOLDS, 'test_folds': TEST_FOLDS,
    'max_per_class': MAX_PER_CLASS, 'efficientat_checkpoint': EA_CHECKPOINT,
    'efficientat_revision': EA_REVISION, 'efficientat_device': EA_DEVICE,
    'yamnet_device': 'cpu', 'threads': THREADS, 'sources': SOURCES,
    'metadata_sha256': hashlib.sha256(metadata_path.read_bytes()).hexdigest(),
    'yamnet_sha256': hashlib.sha256(YAMNET_PATH.read_bytes()).hexdigest(),
    'versions': versions,
    'yamnet_clip_aggregation': 'mean_over_overlapping_windows',
    'efficientat_clip_aggregation': 'full_clip',
    'category_aggregation': 'max_over_native_labels',
}
(RESULTS / 'configuration.json').write_text(json.dumps(config, ensure_ascii=False, indent=2))
print('Prédictions enregistrées', len(predictions))


## 8. Scores de classification sur les huit catégories cibles

L'accuracy et le macro-F1 ci-dessous concernent uniquement les fichiers cibles du test.
Le modèle choisit parmi les huit catégories, ce qui impose un choix même si tous les scores sont faibles.
Le test avec seuils et sons de fond est présenté à la cellule suivante.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

summary = []
reports = []
for model_name in PREDICTORS:
    test = predictions[(predictions.model == model_name) & (predictions.partition == 'test')]
    target_test = test[test.is_target]
    truth = target_test.reference
    predicted = target_test.prediction_8_classes
    summary.append({
        'model': model_name, 'device': test.device.iloc[0],
        'n_target_test': len(target_test),
        'accuracy': accuracy_score(truth, predicted),
        'macro_f1': f1_score(truth, predicted, labels=CATEGORIES, average='macro', zero_division=0),
        'seconds_per_clip': test.seconds.mean(),
        'p95_seconds_per_clip': test.seconds.quantile(.95),
        'rtf_compute': test.seconds.sum() / test.audio_seconds.sum(),
    })
    report = pd.DataFrame(classification_report(truth, predicted, labels=CATEGORIES, output_dict=True, zero_division=0)).T
    report['model'] = model_name
    reports.append(report)
    print(model_name)
    display(report.round(3))

comparison = pd.DataFrame(summary)
display(comparison.round(4))
comparison.to_csv(RESULTS / 'comparison.csv', index=False)
pd.concat(reports).to_csv(RESULTS / 'classification_reports.csv')
ax = comparison.set_index('model')[['accuracy', 'macro_f1']].plot.bar(figsize=(8, 4), ylim=(0, 1))
ax.set_ylabel('Score')
ax.set_title('Classification sur les huit sons cibles — test')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULTS / 'comparison.png', dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, model_name in zip(axes, PREDICTORS):
    part = predictions[(predictions.model == model_name) & (predictions.partition == 'test') & predictions.is_target]
    matrix = confusion_matrix(part.reference, part.prediction_8_classes, labels=CATEGORIES)
    ConfusionMatrixDisplay(matrix, display_labels=CATEGORIES).plot(ax=ax, xticks_rotation=60, colorbar=False)
    ax.set_title(model_name)
plt.tight_layout()
plt.savefig(RESULTS / 'confusion_matrices.png', dpi=150)
plt.show()


## 9. Régler les seuils sur la calibration et tester les alertes par fichier

Un seuil par modèle et catégorie est choisi pour maximiser le F1 sur les folds 1–2.
Les seuils sont ensuite figés pour le test. Chaque catégorie est décidée indépendamment,
ce qui autorise plusieurs détections. La recherche utilise uniquement les scores des fichiers de calibration.

Les sons de fond couvrent quatre catégories. Le taux de fausse alerte calculé ici concerne
ces fichiers et ne représente pas des fausses alertes par heure en écoute continue.
Un score de modèle n'est pas une probabilité de fiabilité calibrée.


In [ ]:
from sklearn.metrics import precision_recall_fscore_support, precision_recall_curve
threshold_rows = []
alert_rows = []
background_rows = []
for model_name in PREDICTORS:
    calibration = predictions[(predictions.model == model_name) & (predictions.partition == 'calibration')]
    test = predictions[(predictions.model == model_name) & (predictions.partition == 'test')].copy()
    decisions = []
    for category in CATEGORIES:
        truth_cal = (calibration.reference == category).to_numpy()
        scores_cal = calibration[f'score_{category}'].to_numpy()
        precision, recall, candidates = precision_recall_curve(truth_cal, scores_cal)
        f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
        # Seuil le plus élevé en cas d'égalité.
        best = np.flatnonzero(np.isclose(f1_values, f1_values.max()))[-1]
        threshold = float(candidates[best])
        threshold_rows.append({'model': model_name, 'category': category, 'threshold': threshold, 'calibration_f1': float(f1_values[best])})
        decision = (test[f'score_{category}'].to_numpy() >= threshold)
        decisions.append(decision)
        truth_test = (test.reference == category).to_numpy()
        p, r, f, _ = precision_recall_fscore_support(truth_test, decision, average='binary', zero_division=0)
        alert_rows.append({'model': model_name, 'category': category, 'precision': p, 'recall': r, 'f1': f, 'n_positive_test': int(truth_test.sum())})
    decision_matrix = np.array(decisions).T
    background_mask = ~test.is_target.to_numpy()
    any_alert = decision_matrix.any(axis=1)
    background_rows.append({
        'model': model_name, 'n_background_test': int(background_mask.sum()),
        'background_clips_with_alert': int(any_alert[background_mask].sum()),
        'background_false_alert_fraction': float(any_alert[background_mask].mean()),
    })
    test['detected_categories'] = [', '.join(np.array(CATEGORIES)[row]) or 'Aucune cible' for row in decision_matrix]
    test.to_csv(RESULTS / f'{model_name.lower()}_test_alerts.csv', index=False)

thresholds = pd.DataFrame(threshold_rows)
alert_metrics = pd.DataFrame(alert_rows)
background_metrics = pd.DataFrame(background_rows)
thresholds.to_csv(RESULTS / 'thresholds.csv', index=False)
alert_metrics.to_csv(RESULTS / 'alert_metrics.csv', index=False)
background_metrics.to_csv(RESULTS / 'background_metrics.csv', index=False)
display(alert_metrics.round(3))
display(background_metrics.round(3))


## 10. Écouter les audios avec les prédictions complètes

Afficher en priorité les erreurs de classification sur les sons cibles du test.
Mettre ONLY_ERRORS=False pour afficher également les prédictions correctes.


In [ ]:
from IPython.display import Audio, HTML, display
from html import escape
ONLY_ERRORS = True

review = predictions[(predictions.partition == 'test') & predictions.is_target].copy()
review['incorrect'] = review.reference != review.prediction_8_classes
if ONLY_ERRORS:
    filenames = review.loc[review.incorrect, 'filename'].drop_duplicates().head(N_AUDIO_DISPLAY)
else:
    filenames = review.filename.drop_duplicates().head(N_AUDIO_DISPLAY)
if len(filenames) == 0:
    print('Aucune erreur sur les catégories cibles du test. Mets ONLY_ERRORS=False pour écouter des exemples.')
for number, filename in enumerate(filenames, 1):
    rows = review[review.filename == filename]
    reference = rows.reference.iloc[0]
    display(HTML(f'<hr><h3>Audio {number} — {escape(TARGETS[reference]["fr"])}</h3><p>{escape(filename)}</p>'))
    wave, rate = sf.read(ESC_ROOT / 'audio' / filename, dtype='float32')
    display(Audio(wave, rate=rate))
    table = rows[['model', 'prediction_8_classes', 'native_top3']].copy()
    table['prediction_8_classes'] = table.prediction_8_classes.map(lambda c: TARGETS[c]['fr'])
    with pd.option_context('display.max_colwidth', None):
        display(table.style.set_properties(**{'white-space': 'normal', 'text-align': 'left'}))


## 11. Simulation optionnelle sur un fichier découpé

Fenêtres de 2 secondes, actualisées toutes les 0,5 seconde. Les deux modèles reçoivent
la même portion temporelle. Le fichier est déjà disponible en entier, donc cette simulation
ne mesure pas la latence utilisateur. Les seuils de la cellule 9 sont réglés sur des fichiers
de 5 secondes et devront être recalibrés avant usage sur ces fenêtres courtes.


In [ ]:
RUN_WINDOW_DEMO = True
WINDOW_SECONDS = 2.0
HOP_SECONDS = 0.5
DEMO_FILENAME = selected[(selected.partition == 'test') & selected.is_target].iloc[0].filename
if RUN_WINDOW_DEMO:
    timeline = []
    for model_name, predictor in PREDICTORS.items():
        rate = SAMPLE_RATES[model_name]
        samples = load_audio(ESC_ROOT / 'audio' / DEMO_FILENAME, rate)
        window_size = round(WINDOW_SECONDS * rate)
        hop_size = round(HOP_SECONDS * rate)
        for start in range(0, len(samples) - window_size + 1, hop_size):
            scores = grouped_scores(model_name, predictor(samples[start:start + window_size]))
            record = {'model': model_name, 'start': start / rate, 'end': (start + window_size) / rate}
            record.update({category: float(scores[i]) for i, category in enumerate(CATEGORIES)})
            timeline.append(record)
    timeline = pd.DataFrame(timeline)
    timeline.to_csv(RESULTS / 'window_demo.csv', index=False)
    wave, rate = sf.read(ESC_ROOT / 'audio' / DEMO_FILENAME, dtype='float32')
    print(DEMO_FILENAME)
    display(Audio(wave, rate=rate))
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    for ax, model_name in zip(axes, PREDICTORS):
        data = timeline[timeline.model == model_name]
        for category in CATEGORIES:
            ax.plot(data.end, data[category], label=TARGETS[category]['fr'])
        ax.set_title(model_name)
        ax.set_ylabel('Score')
        ax.set_ylim(0, 1)
    axes[-1].set_xlabel('Fin de fenêtre dans le fichier (secondes)')
    axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(RESULTS / 'window_demo.png', dpi=150)
    plt.show()


## 12. Télécharger les résultats

L'archive contient les prédictions, scores, seuils, manifestes, paramètres et graphiques.
Elle exclut les audios et les poids. Utiliser également Save Version pour conserver le notebook et ses sorties.


In [ ]:
from IPython.display import FileLink
archive_path = ROOT.parent / 'HearAssist_resultats_sonores.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.glob('*')):
        if path.is_file():
            bundle.write(path, arcname=path.name)
print('Télécharger les résultats')
display(FileLink(str(archive_path)))
print('Si le lien ne fonctionne pas, utilise le panneau des fichiers de sortie Kaggle.')


## Interprétation et suite

- Examiner le F1 de chaque son et les confusions, en plus de l'accuracy globale.
- Comparer le coût de calcul en conservant la colonne device et les versions exportées.
- Le RTF présenté concerne le calcul sur fichiers, il exclut l'attente d'accumulation des fenêtres.
- Compléter les sons absents et les situations réelles avec des enregistrements indépendants.
- Pour un fine-tuning futur, créer une vraie partition d'entraînement et garder un test indépendant.
- Vérifier l'historique des checkpoints avant de présenter ces résultats comme un test totalement indépendant du préentraînement.

Dépannage

- Erreur de téléchargement → activer Internet et relancer la cellule concernée.
- GPU absent → EA_DEVICE='cpu' ou activer un GPU avant de recommencer.
- Erreur torch/torchvision/torchaudio → repartir d'une session Kaggle neuve avec l'image standard.
  Les trois bibliothèques doivent provenir d'une installation compatible. Ce notebook conserve celles de Kaggle.
- Le premier lancement inclut le téléchargement des données et des poids.

Sources

- [ESC-50](https://github.com/karolpiczak/ESC-50)
- [YAMNet](https://github.com/tensorflow/models/tree/master/research/audioset/yamnet)
- [Modèle YAMNet et intégration audio](https://developers.google.com/edge/mediapipe/solutions/audio/audio_classifier)
- [EfficientAT](https://github.com/fschmid56/EfficientAT)
- [LiteRT](https://developers.google.com/edge/litert/overview)

Validation de ce livrable

La syntaxe de toutes les cellules a été vérifiée. L'inférence des deux modèles, les métriques,
les graphiques et l'export ZIP ont été exécutés localement sur CPU avec un petit lot
de 24 audios réels. Le téléchargement complet d'ESC-50 et l'exécution des 480 fichiers
sur l'environnement Kaggle de l'utilisateur restent à effectuer.
